# ORB Final Validation Notebook

This notebook validates the final focused-campaign recommendation:

- Config: `focused_long_only_rr5_ema30_moderate_band_risk_0p25`
- RR: `5`
- EMA: `EMA30`
- ATR mode: `moderate_band`
- Risk per trade: `0.25%`
- Dataset: `MNQ_1mim.parquet`

It is the strongest relative robustness candidate from the focused branch, but the companion report should still be consulted for the subscription-drag caveat.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent

if not (root / "pyproject.toml").exists():
    raise RuntimeError("Could not locate the repository root from the current working directory.")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Project root: {root}")


In [ ]:
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.analytics.diagnostics import performance_by_month, performance_by_year
from src.analytics.metrics import compute_metrics
from src.analytics.orb_campaign import build_strategy, prepare_base_rth_dataset, prepare_current_logic_dataset, resolve_atr_bounds
from src.config.orb_campaign import ORBExperiment, PropConstraintConfig, build_execution_profiles, build_focused_atr_regimes
from src.engine.backtester import run_backtest
from src.engine.execution_model import ExecutionModel
from src.engine.portfolio import build_equity_curve
from src.visualization.equity import plot_drawdown_curve, plot_equity_curve
from src.visualization.plots import plot_trade_histogram

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)


In [ ]:
DATASET_PATH = root / "data" / "raw" / "MNQ_1mim.parquet"
EXPERIMENT = ORBExperiment(**{'name': 'focused_long_only_rr5_ema30_moderate_band_risk_0p25', 'axis': 'focused_prop_branch', 'group': 'focused_long_only_ema_atr_prop', 'strategy_variant': 'current_orb', 'dataset_key': 'current_1m_rth', 'execution_profile': 'repo_realistic', 'or_minutes': 15, 'target_multiple': 5.0, 'side_mode': 'long_only', 'entry_buffer_ticks': 0, 'stop_buffer_ticks': 2, 'opening_time': '09:30:00', 'time_exit': '16:00:00', 'one_trade_per_day': True, 'risk_per_trade_pct': 0.25, 'initial_capital_usd': 50000.0, 'tick_value_usd': 0.5, 'point_value_usd': 2.0, 'max_leverage': None, 'atr_period': 14, 'atr_regime': 'moderate_band', 'atr_min': None, 'atr_max': None, 'direction_filter_mode': 'ema_only', 'ema_length': 30, 'entry_on_next_open': True, 'notes': 'Focused practical campaign on MNQ_1mim.parquet: long-only current ORB, EMA directional filter, optional ATR regime filter, and prop-style sizing.'})
PROP_CONSTRAINTS = PropConstraintConfig(**{'name': 'topstep_50k_reference', 'account_size_usd': 50000.0, 'max_loss_limit_usd': 2000.0, 'daily_loss_limit_usd': 1000.0, 'profit_target_usd': 3000.0, 'monthly_subscription_cost_usd': 150.0, 'trading_days_per_month': 21.0, 'daily_loss_limit_basis': 'realized_daily_pnl'})

pd.DataFrame({"field": list(asdict(EXPERIMENT).keys()), "value": list(asdict(EXPERIMENT).values())})


In [ ]:
base_rth_df = prepare_base_rth_dataset(DATASET_PATH)
current_df = prepare_current_logic_dataset(
    base_rth_df,
    ema_lengths=[EXPERIMENT.ema_length] if EXPERIMENT.ema_length is not None else [],
    include_vwap=False,
)
atr_bounds = resolve_atr_bounds(current_df, build_focused_atr_regimes())
atr_min, atr_max = atr_bounds[EXPERIMENT.atr_regime]
experiment = replace(EXPERIMENT, atr_min=atr_min, atr_max=atr_max)

profiles = build_execution_profiles()
profile = profiles[experiment.execution_profile]
execution_model = ExecutionModel(
    commission_per_side_usd=profile.commission_per_side_usd,
    slippage_ticks=profile.slippage_ticks,
    tick_size=profile.tick_size,
)

strategy = build_strategy(experiment, tick_size=profile.tick_size)
signal_df = strategy.generate_signals(current_df)
session_dates = pd.Index(pd.to_datetime(current_df["session_date"]).dt.date.unique())

trades = run_backtest(
    signal_df,
    execution_model=execution_model,
    tick_value_usd=experiment.tick_value_usd,
    point_value_usd=experiment.point_value_usd,
    time_exit=experiment.time_exit,
    stop_buffer_ticks=experiment.stop_buffer_ticks,
    target_multiple=experiment.target_multiple,
    account_size_usd=experiment.initial_capital_usd,
    risk_per_trade_pct=experiment.risk_per_trade_pct,
    entry_on_next_open=experiment.entry_on_next_open,
    max_leverage=experiment.max_leverage,
)

metrics = compute_metrics(
    trades,
    signal_df=signal_df,
    session_dates=session_dates,
    initial_capital=experiment.initial_capital_usd,
    prop_constraints=PROP_CONSTRAINTS,
)
equity = build_equity_curve(trades, initial_capital=experiment.initial_capital_usd)


In [ ]:
key_metric_order = [
    "n_trades",
    "win_rate",
    "avg_win",
    "avg_loss",
    "expectancy",
    "profit_factor",
    "cumulative_pnl",
    "max_drawdown",
    "sharpe_ratio",
    "longest_loss_streak",
    "average_loss_streak_length",
    "count_loss_streaks_2_plus",
    "worst_trade",
    "worst_day",
    "stop_hit_rate",
    "target_hit_rate",
    "eod_exit_rate",
    "percent_of_days_traded",
    "days_to_profit_target",
    "estimated_months_to_profit_target",
    "profit_target_reached_before_max_loss",
    "any_daily_loss_limit_breach",
    "number_of_daily_loss_limit_breaches",
    "subscription_drag_estimate",
    "estimated_pnl_after_subscription",
]

metrics_table = pd.DataFrame([{"metric": metric, "value": metrics.get(metric)} for metric in key_metric_order])
metrics_table


In [ ]:
if trades.empty:
    print("No trades were generated for the selected configuration.")
else:
    plot_equity_curve(equity)
    plot_drawdown_curve(equity)
    plt.show()

    plot_trade_histogram(trades)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.hist(trades["net_pnl_usd"], bins=30)
    plt.title("Trade PnL Distribution")
    plt.xlabel("Net PnL (USD)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


In [ ]:
def loss_streak_lengths(pnl: pd.Series) -> list[int]:
    streaks = []
    current = 0
    for value in pnl:
        if value < 0:
            current += 1
        elif current > 0:
            streaks.append(current)
            current = 0
    if current > 0:
        streaks.append(current)
    return streaks

streaks = loss_streak_lengths(trades["net_pnl_usd"]) if not trades.empty else []
if not streaks:
    print("No losing streaks to plot.")
else:
    bins = range(1, max(streaks) + 2)
    plt.figure(figsize=(8, 4))
    plt.hist(streaks, bins=bins, align="left", rwidth=0.85)
    plt.title("Losing Streak Distribution")
    plt.xlabel("Consecutive losses")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


In [ ]:
if trades.empty:
    print("No monthly or yearly breakdown available.")
else:
    monthly = performance_by_month(trades)
    yearly = performance_by_year(trades)
    display(monthly.tail(24))
    display(yearly)


In [ ]:
def plot_sample_trade_windows(price_df: pd.DataFrame, trades_df: pd.DataFrame, sample_count: int = 3) -> None:
    if trades_df.empty:
        print("No sample trades to visualize.")
        return

    ordered = trades_df.sort_values("entry_time").reset_index(drop=True)
    positions = sorted(set(np.linspace(0, len(ordered) - 1, num=min(sample_count, len(ordered)), dtype=int)))

    for position in positions:
        trade = ordered.iloc[position]
        window = price_df.loc[
            (price_df["timestamp"] >= trade["entry_time"] - pd.Timedelta(minutes=20))
            & (price_df["timestamp"] <= trade["exit_time"] + pd.Timedelta(minutes=40))
        ].copy()
        if window.empty:
            continue

        plt.figure(figsize=(12, 4))
        plt.plot(window["timestamp"], window["close"], label="Close", linewidth=1.25, color="#111827")
        plt.axvline(trade["entry_time"], color="#0f766e", linestyle="--", label="Entry")
        plt.axvline(trade["exit_time"], color="#b91c1c", linestyle="--", label="Exit")
        plt.axhline(trade["entry_price"], color="#0f766e", linewidth=1.0, alpha=0.8, label="Entry price")
        plt.axhline(trade["stop_price"], color="#dc2626", linewidth=1.0, alpha=0.8, label="Stop")
        plt.axhline(trade["target_price"], color="#2563eb", linewidth=1.0, alpha=0.8, label="Target")

        nearest_idx = (window["timestamp"] - trade["entry_time"]).abs().idxmin()
        entry_bar = window.loc[nearest_idx]
        if pd.notna(entry_bar.get("or_high")):
            plt.axhline(entry_bar["or_high"], color="#7c3aed", linewidth=0.9, alpha=0.6, label="OR high")
        if pd.notna(entry_bar.get("or_low")):
            plt.axhline(entry_bar["or_low"], color="#f59e0b", linewidth=0.9, alpha=0.6, label="OR low")

        plt.title(
            f"Trade {int(trade['trade_id'])} | {trade['session_date']} | {trade['direction']} | {trade['exit_reason']}"
        )
        plt.ylabel("Price")
        plt.xticks(rotation=45)
        plt.legend(loc="best")
        plt.tight_layout()
        plt.show()

plot_sample_trade_windows(current_df, trades, sample_count=3)


## Conclusion

This configuration was selected because it led the focused robustness leaderboard, kept max drawdown at `-1110.50`, and preserved a practical trade count of `332`. The main caveat from the campaign is still time-to-target: visual validation here is about confirming the best relative setup in the tested branch, not claiming that the branch is already subscription-efficient for a prop evaluation.
